# 1 · A contact card

**The problem.** You are writing a contacts service. A contact is a handful of text fields today, but the same data
has to be read by a Python back end, stored as JSON, edited by hand as YAML, and checked before it is trusted. In
plain TypeScript you would write an `interface`. It disappears at runtime, though, so each of those needs adds a
separate description of the same record: a JSON Schema, a Python model, a validation library. Sooner or later the
copies disagree.

This framework takes a different route. **The description of the data is itself data.** You build a *schema* once,
at runtime, with a small fluent DSL. Everything else works from that one schema value: instances, serialization,
validation, and in time generated code for other languages. The Python binding has the same API, down to the method
names.

This first case study builds a single record and covers the handful of ideas every later one relies on:

1. Schemas are values, built with *builders*.
2. Instances are built with builders too. They are read-only once built.
3. A builder ends in one of `create()`, `clone()` or `update()`.
4. A *Spec* is either a value or a function that configures a builder.
5. Every property is optional. "Not set" is a real state, distinct from `""`, `0n` or `false`.

In [1]:
import { Proxies, Schemas } from "@mbse/schemas/Framework";

const store = new Proxies.OfStore();

## Describing the record

A schema for an object with properties is an `OfObject`. You get one from its builder:
`new Schemas.OfObject.Builder()`. Each property is described by a function that receives a *property builder* (`p`),
names the property and gives its type. The type is described by a function too: it receives an "any schema" builder
(`t`) and picks a kind. Here every kind is `as_native(...)`: a value native to the language. In TypeScript the
native types are named by their constructors: `String`, `BigInt` (integers), `Number` (floats), `Boolean` and
`Uint8Array` (bytes).

Read the chain from left to right. It says what it builds, and the final `.create()` hands you the finished schema.

In [2]:
const Contact = new Schemas.OfObject.Builder().ref()
  .properties(
    (p) => p.name("given_name").of((t) => t.as_native(String)),
    (p) => p.name("family_name").of((t) => t.as_native(String)),
    (p) => p.name("birth_date").of((t) => t.as_native(String)), // ISO 8601, e.g. '1852-05-04'
    (p) => p.name("vip").of((t) => t.as_native(Boolean)),
    (p) => p.name("photo").of((t) => t.as_native(Uint8Array)),
  )
  .create();
console.log(Contact);

ObjectData {
  properties: Map(5) {
    "given_name" => NativeData { type: [Function: String] },
    "family_name" => NativeData { type: [Function: String] },
    "birth_date" => NativeData { type: [Function: String] },
    "vip" => NativeData { type: [Function: Boolean] },
    "photo" => NativeData { type: [Function: Uint8Array] }
  },
  adjacencies: Map(0) {},
  singleton: null
}


### Why not an interface or a class?

A TypeScript interface is erased when the code runs, and a class is only meaningful to JavaScript. The schema above
is a plain data structure. It can be inspected, compared and printed at runtime, and (once meta-schemas land)
serialized and turned into code for other languages.

The arrow functions are only a way to *configure* a builder in place. They don't end up in the schema. Each receives
a builder, calls fluent methods on it, and returns it. This "function that configures a builder" is called a
**Spec**, and it appears everywhere in the framework. Because the schema builders are fully typed, TypeScript infers
`p` and `t` for you.

A Spec can also be a finished value. `as_native(String)` passes the type directly. An existing schema can be passed
wherever a Spec is expected, and it is reused, not copied:

In [3]:
const Text = new Schemas.OfNative.Data(String);

const direct = new Schemas.OfAny.Builder().as_native(String).create();
const configured = new Schemas.OfAny.Builder().as_native((n) => n.type(String)).create();
console.log(direct.equals(configured) && configured.equals(Text));

const Named = new Schemas.OfObject.Builder()
  .properties(
    (p) => p.name("first").of(Text), // a value Spec: an existing schema, reused
    (p) => p.name("last").of((t) => t.as_native(String)), // a callable Spec: built in place
  )
  .create();
console.log(Named.properties.get("first") === Text, Text.equals(Named.properties.get("last")));

true


true true


Schema equality is structural and is spelled `.equals()`, since JavaScript has no operator overloading. Use `===`
when you mean "the very same schema object".

## Checking the schema, when you ask

Building never validates. That lets you build schemas in steps: case study 2 needs a moment when two schemas refer
to each other and neither is complete yet. When you want to know whether a schema makes sense, ask it:

In [4]:
console.log(Contact.validate());

const Broken = new Schemas.OfObject.Builder().properties((p) => p.name("age").of((t) => t.as_native(Array))).create();
console.log(Broken.validate());

[]


[ "property 'age': unsupported native type <class 'list'>" ]


`validate()` returns a list of problems rather than throwing, so a tool can report all of them at once. The
messages use the framework's own type names (`list`, `int`, `str`), the same in every language binding.

## Making contacts

To make instances, register the schema under a global name. The name is how other programs, and serialized data,
will refer to this kind of object:

In [5]:
store.register("Contact", Contact);
const Builders = store;

`Builders.Contact()` returns an *instance builder*. It has one fluent setter per property, generated from the schema
at runtime. That is the "dynamic" implementation, *proxies*: no code generation step, and the schema drives
everything.

Because the setters are created at runtime, TypeScript can't know their names. Proxy builders and instances are
therefore typed loosely (their properties are `any`). The planned generated bindings will give each schema its own
precise types; the proxies trade that for needing no build step.

In [6]:
const alice = Builders.Contact()
  .given_name("Alice")
  .family_name("Liddell")
  .birth_date("1852-05-04")
  .vip(false)
  .create();
console.log(alice.given_name, alice.family_name, alice.birth_date, alice.vip);

Alice Liddell 1852-05-04 false


## Instances are read-only

Properties read like fields, but you can't assign to them. TypeScript already rejects the assignment (instances are
typed `readonly`), and at runtime the proxy refuses it too, for code that gets past the compiler:

In [7]:
try {
  // @ts-expect-error: instance properties are read-only
  alice.given_name = "Alicia";
} catch (error) {
  console.log(String(error));
}

AttributeError: proxy properties are read-only; use a builder


**Why?** An object here is more than its fields. In case study 2 it gains *relations* to other objects, and a change
can touch several objects at once. Routing every change through a builder gives one place where a change is
assembled and one moment when it takes effect: `update()`. It also keeps the door open for recording changes as data
(*mutations*) later, without changing how you write code.

## Changing things: `update()` and `clone()`

Pass an existing instance to the builder. The builder starts from a copy of that instance's state. Then finish
with:

- `update()` to write the changes back into that same instance, or
- `clone()` to produce a new instance and leave the original untouched.

`create()` is only for builders that started empty. Each ending is valid in exactly one situation, so a builder
can never quietly do the wrong one.

In [8]:
const same = Builders.Contact(alice).family_name("Hargreaves").update();
console.log(same === alice, alice.family_name);

const sister = Builders.Contact(alice).given_name("Lorina").clone();
console.log(sister === alice, sister.given_name, sister.family_name, "| original:", alice.given_name);

true Hargreaves


false Lorina Hargreaves | original: Alice


In [9]:
for (const wrong of [() => Builders.Contact(alice).create(), () => Builders.Contact().update()]) {
  try {
    wrong();
  } catch (error) {
    console.log(String(error));
  }
}

ValueError: create() is only valid without a source instance; use clone() or update()


ValueError: update() is only valid with a source instance


## "Not set" is a state of its own

Every property is optional: nothing is mandatory unless a constraint says so. A property that was never set is
*absent*, and reading it throws. It doesn't quietly return `undefined`:

In [10]:
try {
  console.log(alice.photo);
} catch (error) {
  console.log(String(error));
}

AttributeError: property 'photo' is not set


**Why not `undefined`?** Because `undefined` would be a value, and then "the photo is undefined" and "nobody said
anything about the photo" would look the same. They mean different things. In a partial update, for example, one
means "remove it" and the other means "leave it alone". Likewise `vip(false)` is a *present* `false`, not an absent
flag.

To test for presence without throwing, use `in`:

In [11]:
console.log("vip" in alice, "photo" in alice, "photo" in alice ? alice.photo : "no photo");

true false no photo


To make a property absent again, *clear* it. This is the other form of Spec, a function. Every setter accepts either
the value or a function that receives a *value handle* `v`. `v.set(...)` and `v.clear()` are the two things you can
do to it. So `.vip(true)` is shorthand for `.vip((v) => v.set(true))`.

Since proxy builders are typed loosely, TypeScript can't infer `v` here, so it's annotated `any`. You'll see this on
every callback given to a *proxy* builder (never on schema builders, which are fully typed).

In [12]:
Builders.Contact(alice).vip((v: any) => v.set(true)).birth_date((v: any) => v.clear()).update();
console.log(alice.vip, "birth_date" in alice);

true false


## Builders don't check types

The instance builder accepts any value for any property. That's deliberate, and it's the same rule as for schemas:
building is cheap and permissive, and checking is something you ask for. Here the photo is set to a string where the
schema says `Uint8Array`:

In [13]:
const sloppy = Builders.Contact().given_name("Bob").photo("not bytes").create();
console.log(sloppy.photo);

not bytes


Nothing complained yet. Case study 5 shows the two places that do: the validator, which reports every problem with a
path, and the serializers, which refuse to write data that doesn't match its schema.

## What you learned

- A schema is a value built with `new Schemas.OfObject.Builder()...create()`. `validate()` checks it when you ask.
- A Spec is a value, or a function that configures a builder and returns it.
- `Proxies.register(name, schema)` makes `Proxies.Builders.<name>()` available. Instances are read-only.
- Builders end with `create()` (from nothing), `update()` (in place) or `clone()` (a copy).
- Properties are optional. Absent isn't `undefined`, and `v.clear()` makes a property absent.
- Proxies are dynamic, so their callbacks take `(x: any)`; schema builders are fully typed.

**Next:** a contact rarely has just one address. Case study 2 introduces *relations*, the framework's only
collection type, and explains why there are no arrays.